# Netflix Titles — Exploratory Data Analysis

An exploratory data analysis (EDA) of the Netflix catalogue using **NumPy**, **Pandas** and **Matplotlib**.

The dataset covers roughly **8,800 movies and TV shows** available on Netflix up to 2021, with their release year, country, genre, maturity rating and duration.

### Questions this notebook answers

1. How recent is the catalogue? Which era dominates?
2. Are movies or TV shows more common?
3. Which countries produce the most content?
4. Which genres stand out?
5. Which age group is the catalogue aimed at?

**Data source:** [Netflix Movies and TV Shows](https://www.kaggle.com/datasets/shivamb/netflix-shows) by shivamb
**Source code / repository:** [github.com/kapollox/netflix-veri-analizi](https://github.com/kapollox/netflix-veri-analizi)

## Setup and loading the data

Each row of the raw catalogue file represents a single movie or TV show.

The loader below searches `/kaggle/input` first and falls back to the working directory,
so the notebook runs both on Kaggle and locally. On Kaggle the dataset must be attached
via **Add Input → Netflix Movies and TV Shows (shivamb/netflix-shows)**.

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_dataset():
    """Locate netflix_titles.csv on Kaggle or in the working directory."""
    candidates = (
        ["/kaggle/input/netflix-shows/netflix_titles.csv"]
        + sorted(glob.glob("/kaggle/input/**/netflix_titles.csv", recursive=True))
        + ["netflix_titles.csv"]
    )
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        "netflix_titles.csv not found. On Kaggle, open the right-hand panel and use "
        "Add Input to attach the 'Netflix Movies and TV Shows' dataset (shivamb/netflix-shows)."
    )


CSV_PATH = find_dataset()
print("Reading:", CSV_PATH)

df = pd.read_csv(CSV_PATH)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Column names:", list(df.columns))

8,807 titles across 12 columns. Most columns hold text — `release_year` is the only numeric field.

---

# Part 1 — Release year statistics (NumPy)

## 1.1 The release year array

The release year column is pulled into a NumPy array for fast vectorised computation.

In [ ]:
years = np.array(df["release_year"])

print("Array type:", type(years))
print("Data type:", years.dtype)
print("Number of elements:", len(years))
print("First 10 values:", years[:10])

The column has no missing values and every entry is an integer, so no cleaning is needed here.

## 1.2 Basic statistics

Centre and spread of the distribution: mean, median, standard deviation and the extremes.

In [ ]:
mean_year = np.mean(years)
median_year = np.median(years)
std_year = np.std(years)
min_year = np.min(years)
max_year = np.max(years)

print("Mean:", round(mean_year, 2))
print("Median:", median_year)
print("Standard deviation:", round(std_year, 2))
print("Minimum:", min_year)
print("Maximum:", max_year)

The mean (2014.18) sits below the median (2017), so the distribution is **left-skewed**:
a small number of very old titles drags the average down.

## 1.3 How recent is the catalogue?

Boolean indexing gives the share of titles released after 2015.

In [ ]:
after_2015 = (years > 2015).sum()
share = after_2015 / len(years) * 100

print("Titles released after 2015:", after_2015)
print("Share: %", round(share, 1))

About two thirds of the catalogue is post-2015 — Netflix clearly leans on recent content.

## 1.4 Distribution across decades

A histogram over ten-year bins.

In [ ]:
bins = np.arange(1920, 2030, 10)
counts, edges = np.histogram(years, bins=bins)

for i in range(len(counts)):
    print(edges[i], "-", edges[i + 1] - 1, ":", counts[i], "titles")

Almost the entire catalogue is concentrated in the 2010–2019 decade; everything before that is sparse.

## 1.5 Which release years are represented?

In [ ]:
distinct_years = np.unique(years)

print("Number of distinct release years:", len(distinct_years))
print("Oldest year:", distinct_years[0], "- Newest year:", distinct_years[-1])

The range spans 1925–2021 (96 years) but only 74 distinct years appear:
some years contribute no titles at all.

---

# Part 2 — Catalogue structure (Pandas)

## 2.1 First look at the dataset

Sample rows, column types and the numeric summary.

In [ ]:
print("First 10 rows:")
print(df.head(10))

In [ ]:
df.info()  # info() prints by itself, so it is not wrapped in print()

In [ ]:
print(df.describe())

`describe()` only reports `release_year`, because that is the single numeric column.

## 2.2 Missing value check

A data quality pass: which columns have gaps, and how large are they?

In [ ]:
missing_values = df.isnull().sum()

print("Missing values per column:")
print(missing_values)

`director` has by far the most gaps. That is expected rather than broken data —
TV shows usually do not credit a single director.

## 2.3 Movies vs. TV shows

What is the catalogue actually made of?

In [ ]:
type_counts = df["type"].value_counts()

print(type_counts)

6,131 movies against 2,676 TV shows — roughly **70% of the catalogue is film**.

## 2.4 A movies-only table

Duration analysis only makes sense for movies, so they get their own table.

In [ ]:
movies_df = df[df["type"] == "Movie"].copy()  # copy() because a new column will be added

print("Rows in movies_df:", len(movies_df))
print(movies_df[["title", "release_year", "duration"]].head())

The row count matches the Movie count above exactly, so the filter works as intended.

## 2.5 Production geography — top 10 countries

In [ ]:
top10_countries = df["country"].value_counts().head(10)

print(top10_countries)

The United States leads by a wide margin. Note that co-productions such as
`"United States, India"` are counted as their own category here; a strict per-country
total would require splitting the column first.

## 2.6 Most popular genres

A title can belong to several genres, so the column is split on commas and
expanded into separate rows with `explode`.

In [ ]:
genres = df["listed_in"].dropna().str.split(", ").explode()
top5_genres = genres.value_counts().head(5)

print(top5_genres)

*International Movies* and *Dramas* lead the list — Netflix's emphasis on
international content shows up clearly here.

## 2.7 Output volume by year

In [ ]:
by_year = df.groupby("release_year").size()
peak_year = by_year.idxmax()
peak_count = by_year.max()

print(by_year)
print("Year with the most titles:", peak_year, "-", peak_count, "titles")

The peak is 2018 (1,147 titles). The decline afterwards is not a real contraction —
the dataset was collected in 2021, so the final years are incomplete.

## 2.8 Movie durations

Text values such as `"90 min"` are converted to numbers. Movie durations always
follow the `"<number> min"` format, so only the unit needs stripping.

In [ ]:
movies_df["duration_minutes"] = movies_df["duration"].str.replace(" min", "", regex=False).astype(float)
longest_5 = movies_df.nlargest(5, "duration_minutes")[["title", "duration", "release_year"]]

print("5 longest movies:")
print(longest_5)
print("Average movie duration:", round(movies_df["duration_minutes"].mean(), 1), "minutes")

At 99.6 minutes the average is close to a standard feature length. Titles running past
300 minutes are exceptions — documentaries, concert films or interactive productions
such as *Black Mirror: Bandersnatch*.

---

# Part 3 — Visualisation (Matplotlib)

## Chart 1 — Movies vs. TV shows

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(type_counts.index, type_counts.values, color="red")

plt.title("Netflix Movie and TV Show Counts")
plt.xlabel("Content Type")
plt.ylabel("Number of Titles")

plt.show()

The movie bar is roughly twice the height of the TV show bar.

## Chart 2 — Output volume over time

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(by_year.index, by_year.values, color="red")

plt.title("Number of Titles by Release Year")
plt.xlabel("Release Year")
plt.ylabel("Number of Titles")

plt.show()

The line is almost flat until the 2000s, then climbs steeply after 2015 and peaks in 2018 —
the growth curve of Netflix's content investment.

## Chart 3 — Top 10 producing countries

Country names are long, so a horizontal bar chart is used instead of a vertical one.

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(top10_countries.index, top10_countries.values, color="red")

plt.title("Top 10 Content-Producing Countries")
plt.xlabel("Number of Titles")
plt.ylabel("Country")
plt.gca().invert_yaxis()  # largest value on top

plt.tight_layout()
plt.show()

The United States dominates with over 2,800 titles — roughly three times India in second place.
South Korea and Japan making the top five reflects the investment in Asian content.

## Chart 4 — Audience profile: maturity rating distribution

There are too many rating categories for a readable pie chart, so the top 6 are kept
and the rest are grouped into "Other".

In [ ]:
rating_counts = df["rating"].value_counts()

pie_data = rating_counts.head(6)
pie_data["Other"] = rating_counts[6:].sum()

plt.figure(figsize=(8, 8))
plt.pie(pie_data.values, labels=pie_data.index, autopct="%1.1f%%")

plt.title("Maturity Rating Distribution")

plt.show()

TV-MA (36.4%) and TV-14 (24.5%) together take up more than 60% of the pie:
the catalogue targets adult and young-adult viewers. Kids' content (TV-Y7) stays at 3.8%.

## Chart 5 — All four findings on one dashboard

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Netflix Data Analysis Summary")

# Top left: content type
axs[0, 0].bar(type_counts.index, type_counts.values, color="red")
axs[0, 0].set_title("Movie and TV Show Counts")
axs[0, 0].set_xlabel("Content Type")
axs[0, 0].set_ylabel("Number of Titles")

# Top right: titles by year
axs[0, 1].plot(by_year.index, by_year.values, color="red")
axs[0, 1].set_title("Number of Titles by Release Year")
axs[0, 1].set_xlabel("Release Year")
axs[0, 1].set_ylabel("Number of Titles")

# Bottom left: top 10 countries
axs[1, 0].barh(top10_countries.index, top10_countries.values, color="red")
axs[1, 0].set_title("Top 10 Content-Producing Countries")
axs[1, 0].set_xlabel("Number of Titles")
axs[1, 0].set_ylabel("Country")
axs[1, 0].invert_yaxis()

# Bottom right: rating distribution
axs[1, 1].pie(pie_data.values, labels=pie_data.index, autopct="%1.1f%%")
axs[1, 1].set_title("Rating Distribution")

plt.tight_layout()
plt.show()

---

# Conclusions

| Finding | Value |
|---|---|
| Total titles | 8,807 |
| Movies / TV shows | 6,131 / 2,676 (70% movies) |
| Most productive year | 2018 — 1,147 titles |
| Titles released after 2015 | 5,656 (64.2%) |
| Mean release year | 2014.18 (median 2017) |
| Leading country | United States, 2,800+ titles |
| Most popular genre | International Movies |
| Dominant rating | TV-MA + TV-14 = 60.9% |
| Average movie duration | 99.6 minutes |
| Longest title | *Black Mirror: Bandersnatch* — 312 minutes |

**Overall:** the Netflix catalogue is US-centric, grew rapidly after 2015, is
movie-heavy and aimed at adult viewers. The mean release year sitting below the
median confirms a left-skewed distribution — a handful of old classics inside a
large body of recent content.

## Notes and limitations

- The `country` column stores co-productions as a single string such as `"United States, India"`.
  In the country ranking these combinations count as separate categories; an exact per-country
  total would require splitting the column.
- The high missing rate in `director` is not bad data — TV shows generally do not credit one director.
- Duration analysis covers movies only; for TV shows the `duration` field holds a season count.
- The dataset was collected in 2021 and does not cover anything released after that.

---

*Source code and the standalone Python script are available at
[github.com/kapollox/netflix-veri-analizi](https://github.com/kapollox/netflix-veri-analizi).
If you found this notebook useful, an upvote is appreciated.*